In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [4]:
def make_map(files, pole='north', q=0, mu_thr=0.1, binning=4):

    nx, ny = 1024, 1024
    xc, yc = 511.5, 511.5
    Rsun = 480


    if pole == 'north':
        view_new = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=q * 3600) ### North pole view
    else:
        view_new = View(nx, ny, xc, yc, Rsun, 90, -90, 0, rsun_arc=q * 3600) ### South pole view

    grid = view_new.grid()
    mean, variance, coverage, coverage2 = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        with fits.open(file) as hdul:
            header = hdul[1].header.copy()
            data = hdul[1].data.copy()

        data = rebin(data, binning, update_header=header)
        view = View.from_header(header)

        transform = (view_new.to_carrington(correct_mu=True) -
                     view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=mu_thr))
        grid_, alpha = transform(grid)
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4
        coverage += np.nan_to_num(n)
        coverage2 += np.nan_to_num(n ** 2)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / coverage)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)


    variance = variance / coverage
    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    coverage2[coverage2 == 0] = np.nan
    variance[coverage == 0] = np.nan

    return mean, variance, coverage, coverage2, view_new

In [5]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/nonpolar/*'))
print(len(files))
print(files[0], files[-1])

357
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_000000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241130_220000_TAI.3.magnetogram.fits


In [6]:
mean, variance, coverage, coverage2, view = make_map(files, pole='north')

/tmp/ipykernel_23572/3707330354.py:37: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [7]:
show_data(mean, view, label=r'$B_{los}$', vmin=-100, vmax=100)

In [8]:
variance_ = variance * coverage2 / coverage ** 2
mean_ = mean.copy()
mean_[np.abs(mean) < 3 * np.sqrt(variance_)] = np.nan

/tmp/ipykernel_23572/3052948849.py:3: RuntimeWarning: invalid value encountered in sqrt
  mean_[np.abs(mean) < 3 * np.sqrt(variance_)] = np.nan


In [9]:
show_data(mean_, view, label=r'$B_{los}$', vmin=-100, vmax=100)

In [10]:
show_data(np.sqrt(variance), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=40)

/tmp/ipykernel_23572/2739061088.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_data(np.sqrt(variance), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=40)


In [13]:
show_data(np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=10)

/tmp/ipykernel_23572/1909495063.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_data(np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=10)


In [12]:
show_data(np.abs(mean) / np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=5)

/tmp/ipykernel_23572/3723752687.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_data(np.abs(mean) / np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=5)
